In [ ]:
#| default_exp core

# Core API

> One-call functions over the pure-python engine

Each call opens the default collection, does its work, and closes up, so there's no session state to manage. `fb` variants take plain `front`/`back` strings and scalar ids.

In [ ]:
from fastcore.utils import *
from fastanki.schema import *
from fastanki.collection import *
from fastanki.syncer import *
from fastanki.scheduler import *

In [ ]:
import tempfile
from fastcore.test import *

In [ ]:
os.environ['FASTANKI_DIR'] = tempfile.mkdtemp()

This notebook runs against a throwaway data folder; without that override, everything below uses `~/.fastanki`.

## Cards and notes

In [ ]:
def add_card(
    model:str='Basic', # Notetype name (the tool description lists each notetype's fields)
    deck:str='Default', # Deck name (`::` for nesting; created if needed)
    tags:str=None, # Space-separated tags
    fields:dict=None, # Field name -> value, e.g. {'Front':'2+2', 'Back':'4'}
):
    "Add a card of any notetype, returning the new note id. Field names must match the notetype."
    with Collection.open() as col:
        return col.add(model=model, deck=deck, tags=tags.split() if tags else None, **(fields or {})).id

def add_fb_card(
    front:str, # Front (question) text
    back:str, # Back (answer) text
    deck:str='Default', # Deck name (`::` for nesting; created if needed)
    tags:str=None, # Space-separated tags
):
    "Add a Basic card, returning the new note id."
    return add_card(deck=deck, tags=tags, fields={'Front':front, 'Back':back})

def add_cloze_card(
    text:str, # Cloze text with `{{c1::hidden}}` deletions
    back_extra:str='', # Extra info shown on the back of every card
    deck:str='Default', # Deck name (`::` for nesting; created if needed)
    tags:str=None, # Space-separated tags
):
    "Add a Cloze card (`{{c1::hidden}}` syntax), returning the new note id."
    return add_card(model='Cloze', deck=deck, tags=tags, fields={'Text':text, 'Back Extra':back_extra})

In [ ]:
nid = add_fb_card('What is the capital of France?', 'Paris', tags='geo')
note = add_card(deck='Spanish::Vocab', tags='spanish', fields={'Front':'hola','Back':'hello'})
note

1784903934511

In [ ]:
czid = add_cloze_card('Minus times {{c1::minus}} is {{c2::plus}}', tags='maths')
test_eq(type(czid), int)

## Finding, updating, removing

In [ ]:
def find_notes(deck:str=None, # Deck name (matches subdecks too)
               tag:str=None, # Tag to match
               added_days:int=None, # Only notes added in the last this-many days
               fields:dict=None): # Field name -> case-insensitive substring, e.g. {'Front':'hello'}
    "Notes matching all given criteria."
    with Collection.open() as col: return col.find_notes(deck=deck, tag=tag, added_days=added_days, **(fields or {}))

def find_note_ids(deck:str=None, # Deck name (matches subdecks too)
                  tag:str=None, # Tag to match
                  added_days:int=None, # Only notes added in the last this-many days
                  fields:dict=None): # Field name -> case-insensitive substring, e.g. {'Front':'hello'}
    "Ids of notes matching all given criteria."
    return [n.id for n in find_notes(deck=deck, tag=tag, added_days=added_days, fields=fields)]

In [ ]:
def find_cards(deck:str=None, # Deck name (matches subdecks too)
               tag:str=None, # Tag to match
               added_days:int=None, # Only cards added in the last this-many days
               is_due:bool=None, # Only cards due for review
               fields:dict=None): # Field name -> case-insensitive substring, e.g. {'Front':'hello'}
    "Cards matching all given criteria."
    with Collection.open() as col: return col.find_cards(deck=deck, tag=tag, added_days=added_days, is_due=is_due, **(fields or {}))

def find_card_ids(deck:str=None, # Deck name (matches subdecks too)
                  tag:str=None, # Tag to match
                  added_days:int=None, # Only cards added in the last this-many days
                  is_due:bool=None, # Only cards due for review
                  fields:dict=None): # Field name -> case-insensitive substring, e.g. {'Front':'hello'}
    "Ids of cards matching all given criteria."
    return [c.id for c in find_cards(deck=deck, tag=tag, added_days=added_days, is_due=is_due, fields=fields)]

In [ ]:
def get_note(
    note_id:int, # Id of the note to retrieve
):
    "Retrieve a note by id."
    with Collection.open() as col: return col.get_note(note_id)

In [ ]:
test_eq(find_note_ids(tag='geo'), [nid])
test_eq(find_note_ids(deck='Spanish'), [note])
test_eq(find_note_ids(fields={'Front':'capital'}), [nid])
test_eq(len(find_cards()), 4)
get_note(nid)

<div class="prose" markdown="1">

**Front**: What is the capital of France? | **Back**: Paris | 🏷 geo

</div>

In [ ]:
def update_note(note, tags=None, add_tags=None, **fields):
    "Update fields and/or tags of a `Note` or note id; `tags` replaces, `add_tags` appends."
    with Collection.open() as col: return col.update_note(note, tags=tags, add_tags=add_tags, **fields)

def update_fb_note(
    note_id:int, # Id of the Basic note to update
    front:str='', # New Front text (empty leaves it unchanged)
    back:str='', # New Back text (empty leaves it unchanged)
    tags:str=None, # Space-separated tags, replacing all existing tags
    add_tags:str=None, # Space-separated tags to add, keeping existing ones
):
    "Update a Basic note's front/back and/or tags."
    kw = {}
    if front: kw['Front'] = front
    if back: kw['Back'] = back
    if tags: tags = tags.split()
    if add_tags: add_tags = add_tags.split()
    return update_note(note_id, tags=tags, add_tags=add_tags, **kw)

def del_note(
    notes:list, # Note ids (or `Note` objects) to delete, along with their cards
):
    "Delete note(s) (and their cards) by `Note` or id, singly or in a list."
    with Collection.open() as col: return col.remove_notes(notes)

In [ ]:
n2 = update_note(note, Back='hello!', add_tags='greeting')
test_eq(n2['Back'], 'hello!')
test_eq(n2.tags, ['spanish','greeting'])
test_eq(update_fb_note(nid, back='Paris, France').fields['Back'], 'Paris, France')
test_eq(del_note([nid, czid]), 2)
test_eq(find_note_ids(tag='geo'), [])

## Reviewing

The study loop a client drives: `next_card` picks what to show, `answer_buttons` says what each button would schedule (for display), `answer_card` grades it. Scheduling follows the collection's own settings — SM-2 or FSRS, whichever the user's other clients use — and every answer lands in the revlog, so a later `sync` carries it to AnkiWeb and on to every other device.

In [ ]:
def next_card(
    deck:str=None, # Deck name (subdecks included; default: whole collection)
):
    "The next card due for study, or None when the session is done."
    with Collection.open() as col: return col.next_card(deck)

def answer_buttons(
    card_id:int, # Id of the card being reviewed
):
    "For each ease 1-4, `(next_state, delay_secs)`: what to show on the answer buttons."
    with Collection.open() as col: return col.answer_buttons(card_id)

def answer_card(
    card_id:int,    # Id of the card being reviewed
    ease:int,       # 1=Again 2=Hard 3=Good 4=Easy
    taken_ms:int=0, # Milliseconds spent answering, for the stats
):
    "Answer a due card, updating its schedule and review log."
    with Collection.open() as col: return col.answer_card(card_id, ease, taken_ms=taken_ms)

def due_counts(
    deck:str=None, # Deck name (subdecks included; default: whole collection)
):
    "(new, learning, review) counts due now."
    with Collection.open() as col: return col.due_counts(deck)

Let's demo studying. We initialise three new cards in their own deck, and conduct a session exactly like the one Anki's clients run: keep asking `next_card` until nothing is due.

`answer_buttons` reports what each ease would schedule, in seconds — the numbers a client renders above its buttons — so a two-line formatter turns a card into the reviewer's view of it:

In [ ]:
for f,b in [('perro','dog'),('gato','cat'),('pájaro','bird')]: add_fb_card(f, b, deck='Práctica')

def delay(secs):
    for n,u in [(86400,'d'),(3600,'h'),(60,'m')]:
        if secs>=n: return f'{round(secs/n)}{u}'
    return f'{secs}s'

def show(c):
    btns = ' · '.join(f'{l} {delay(s)}' for l,(_,s) in zip(('Again','Hard','Good','Easy'), answer_buttons(c.id)))
    return f"{get_note(c.nid)['Front']:8} [{btns}]"

show(next_card('Práctica'))

'perro    [Again 1m · Hard 6m · Good 10m · Easy 3d]'

Those are the default learning steps: Again restarts at 1 minute, Good advances to the 10-minute step, Easy skips straight to a 4-day review. If we answer Good all the way through, the session settles in one sitting — when nothing else is waiting, Anki serves learning cards up to 20 minutes early, so each card appears twice (the 1-minute step, then the 10-minute step) and graduates to tomorrow:

In [ ]:
while (c := next_card('Práctica')):
    print(show(c))
    answer_card(c.id, 3)
due_counts('Práctica')

perro    [Again 1m · Hard 6m · Good 10m · Easy 3d]


gato     [Again 1m · Hard 6m · Good 10m · Easy 3d]


pájaro   [Again 1m · Hard 6m · Good 10m · Easy 3d]


gato     [Again 1m · Hard 10m · Good 1d · Easy 3d]


perro    [Again 1m · Hard 10m · Good 1d · Easy 3d]


pájaro   [Again 1m · Hard 10m · Good 1d · Easy 5d]


(0, 0, 0)

In [ ]:
c = find_cards(deck='Práctica')[0]
test_eq((c.type, c.queue, c.ivl), (2, 2, 1))  # graduated: review cards on a 1-day interval

Let's fast-forward: make `perro` a 7-day review card due today. Now the buttons show SM-2's ladder — Hard grows the interval a little and eases off, Good multiplies by the card's ease factor, Easy adds a bonus — and failing it drops the card into relearning, with a lapse on its record and its ease knocked down:

In [ ]:
with Collection.open() as col: col.con.execute('update cards set ivl=7, due=? where id=?', (col.today(), c.id))
print(show(c))
lapsed = answer_card(c.id, 1)
(lapsed.type, lapsed.queue, lapsed.lapses, lapsed.factor)

perro    [Again 10m · Hard 8d · Good 16d · Easy 21d]


(3, 1, 1, 2300)

Every answer lands in the review log: the button pressed, the interval it produced (positive days, negative seconds), the one it replaced, and the kind of review. This is the durable record — `sync` carries it to AnkiWeb, and any other client can rebuild scheduling state from it:

In [ ]:
with Collection.open() as col: rlog = col.q('select ease, ivl, lastIvl, type from revlog where cid=?', c.id)
rlog

[(3, -600, 0, 0), (3, 1, -600, 0), (1, -600, 7, 1)]

Scheduling follows the collection's own configuration. Enable FSRS (the flag your other clients set when you turn it on in deck options) and the very same loop schedules with the memory model from `fastanki.fsrs` instead, no code changes:

In [ ]:
with Collection.open() as col: col.con.execute("insert or replace into config values ('fsrs',-1,0,?)", (b'true',))
show(c)

'perro    [Again 10m · Hard 15m · Good 1d · Easy 2d]'

In [ ]:
#| hide
with Collection.open() as col:
    col.con.execute("delete from config where key='fsrs'")
    col.remove_deck('Práctica')

## Syncing

In [ ]:
def sync(
    user:str=None, # AnkiWeb email (only needed the first time)
    passw:str=None, # AnkiWeb password (only the first time; a host key is saved after)
    endpoint:str=None, # Sync server URL (defaults to AnkiWeb)
    upload:bool=False, # Force-upload the local collection, replacing the server copy
):
    "Sync the default collection with AnkiWeb. Pass credentials the first time; they're saved after that."
    with Collection.open() as col: return col.sync(user=user, passw=passw, endpoint=endpoint, upload=upload)

The first sync of a fresh collection is a full one: by default that's a download (the server copy wins), and replacing a non-empty server copy with a fresh empty collection is refused unless the server side is empty too. Pass `upload=True` deliberately to push your local copy wholesale.

In [ ]:
#| eval: false
sync(user=os.environ['ANKI_USER'], passw=os.environ['ANKI_PASS'])  # first time
sync()  # after that

## Tool use

In [ ]:
def anki_tools(): print('&`[add_card, add_fb_card, add_cloze_card, find_notes, find_note_ids, find_cards, find_card_ids, get_note, del_note, update_fb_note, next_card, answer_buttons, answer_card, due_counts, sync]`')

In [ ]:
anki_tools()

&`[add_card, add_fb_card, add_cloze_card, find_notes, find_note_ids, find_cards, find_card_ids, get_note, del_note, update_fb_note, next_card, answer_buttons, answer_card, due_counts, sync]`


`add_card` handles any notetype via a `fields` dict, and rejects unknown field names with a clear error:

In [ ]:
czid = add_card(model='Cloze', fields={'Text':'{{c1::pi}} ~ 3.14'})
test_eq(get_note(czid)['Text'], '{{c1::pi}} ~ 3.14')
test_fail(lambda: add_card(fields={'Nope':'x'}), contains='Nope')   # unknown field -> clear error